In [2]:
# 1. ติดตั้งไลบรารี (หากยังไม่ได้ติดตั้งใน session นี้ ให้ uncomment บรรทัดล่างครับ)
!pip install -q xee geemap xarray shapely matplotlib pandas

zsh:1: command not found: pip


In [3]:
# ========================================================
# โค้ดวิเคราะห์ไฟป่าด้วย dNBR จากข้อมูล Sentinel-2 (Colab)
# ========================================================
import ee
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import shapely
from xee import helpers
from matplotlib.colors import ListedColormap, BoundaryNorm

# 1. ยืนยันตัวตน GEE (ถ้ายังไม่ได้ทำ)
ee.Authenticate()
ee.Initialize(project='ee-end-to-end-gee')

# ========================================================
# 2. กำหนดพื้นที่ศึกษา (Phitsanulok Area) และช่วงเวลา
# ========================================================
min_lon, min_lat = 100.16119010692728, 16.795978728481742
max_lon, max_lat = 100.23270040751413, 16.860656792384646

roi_ee = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])
aoi_shapely = shapely.geometry.box(min_lon, min_lat, max_lon, max_lat)

# กำหนดช่วงเวลาก่อนและหลังเกิดไฟป่า (ปรับแก้ได้ตามเหตุการณ์จริง)
pre_start, pre_end = '2023-01-01', '2023-01-31'   # ก่อนไฟไหม้ (ม.ค.)
post_start, post_end = '2023-04-20', '2023-05-15' # หลังไฟไหม้ (เม.ย.)

# ========================================================
# 3. ดึงข้อมูล Sentinel-2 และคำนวณ NBR
# ========================================================
print("กำลังประมวลผลข้อมูล Sentinel-2 และคำนวณ dNBR...")

# ฟังก์ชันกำจัดเมฆสำหรับ Sentinel-2
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    # ปรับสเกลข้อมูล (Scale factor) ด้วยการหาร 10,000
    return image.updateMask(mask).divide(10000).copyProperties(image, ["system:time_start"])

# ดึงข้อมูล Sentinel-2 (Level-2A Surface Reflectance)
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi_ee).map(mask_s2_clouds)

# สร้างภาพตัวแทน (Median) ก่อนและหลังไฟไหม้
pre_img = s2.filterDate(pre_start, pre_end).median()
post_img = s2.filterDate(post_start, post_end).median()

# คำนวณ NBR (Band 8 = NIR, Band 12 = SWIR2)
nbr_pre = pre_img.normalizedDifference(['B8', 'B12']).rename('NBR_PRE')
nbr_post = post_img.normalizedDifference(['B8', 'B12']).rename('NBR_POST')

# คำนวณ dNBR = Pre - Post
dnbr = nbr_pre.subtract(nbr_post).rename('dNBR')

# รวมแบนด์ทั้งหมดเป็น 1 ภาพ เพื่อส่งเข้า Xarray
combined = ee.Image.cat([nbr_pre, nbr_post, dnbr])

# ========================================================
# 4. ดึงข้อมูลเข้าสู่ Xarray
# ========================================================
# ใช้สเกล 0.001 (ประมาณ 100 เมตร) เพื่อป้องกัน RAM เต็มในพื้นที่กว้าง
grid_params = helpers.fit_geometry(
    geometry=aoi_shapely,
    grid_crs='EPSG:4326',
    grid_scale=(0.001, -0.001)
)

# ดึงข้อมูลและยุบมิติเวลา (Squeeze) เนื่องจากเรามีแค่ภาพเดียว
ds = xr.open_dataset(
    ee.ImageCollection([combined]),
    engine='ee',
    **grid_params
).squeeze()

# ========================================================
# 5. พล็อตแผนที่ผลลัพธ์
# ========================================================
print("กำลังสร้างแผนที่ผลลัพธ์...")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ภาพที่ 1: NBR ก่อนไฟไหม้
ds.NBR_PRE.plot(ax=axes[0], cmap='Greens', vmin=-0.2, vmax=0.8, cbar_kwargs={'label': 'NBR'})
axes[0].set_title(f'Pre-fire NBR\n({pre_start} to {pre_end})', fontweight='bold')

# ภาพที่ 2: NBR หลังไฟไหม้
ds.NBR_POST.plot(ax=axes[1], cmap='Greens', vmin=-0.2, vmax=0.8, cbar_kwargs={'label': 'NBR'})
axes[1].set_title(f'Post-fire NBR\n({post_start} to {post_end})', fontweight='bold')

# ภาพที่ 3: dNBR (Burn Severity Classification)
# สร้างเกณฑ์สีตามมาตรฐานสากล (USGS Burn Severity)
cmap = ListedColormap(['#008000', '#00fc00', '#ffff00', '#ffaa00', '#ff0000', '#cc0000'])
bounds = [-0.5, -0.1, 0.1, 0.27, 0.66, 1.3]
norm = BoundaryNorm(bounds, cmap.N)

# พล็อตภาพจำแนกระดับความรุนแรง
p = ds.dNBR.plot(ax=axes[2], cmap=cmap, norm=norm, add_colorbar=False)
axes[2].set_title('Burn Severity (dNBR)', fontweight='bold')

# สร้าง Colorbar แบบกำหนดเอง
cbar = fig.colorbar(p, ax=axes[2], boundaries=bounds, ticks=[-0.3, 0, 0.18, 0.46, 0.98])
cbar.ax.set_yticklabels(['Enhanced Regrowth', 'Unburned', 'Low Severity', 'Moderate Severity', 'High Severity'])

plt.suptitle('Wildfire Burn Severity Analysis (Sentinel-2)', y=1.05, fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'ee'